In [1]:
import pandas as pd
import numpy as np 
import scanpy as sc 
import os
import scrublet as scr 
import glob

In [2]:
# find . -type f | sort

In [3]:
study_files = {
    "Hwang2022_Group1": {
        "matrix": "Data_Hwang2022_Pancreas/Group1/Exp_data_TP10K_1.mtx",
        "genes":  "Data_Hwang2022_Pancreas/genes.txt",
        "cells":  "Data_Hwang2022_Pancreas/Group1/Cells1.csv",
        "samples": "Data_Hwang2022_Pancreas/Samples.csv",
        "units": "UMI"
    },
    "Hwang2022_Group2": {
        "matrix": "Data_Hwang2022_Pancreas/Group2/Exp_data_TP10K_2.mtx",
        "genes":  "Data_Hwang2022_Pancreas/genes.txt",
        "cells":  "Data_Hwang2022_Pancreas/Group2/Cells2.csv",
        "samples": "Data_Hwang2022_Pancreas/Samples.csv",
        "units": "UMI"
    },
    "Hwang2022_Group3": {
        "matrix": "Data_Hwang2022_Pancreas/Group3/Exp_data_TP10K_3.mtx",
        "genes":  "Data_Hwang2022_Pancreas/genes.txt",
        "cells":  "Data_Hwang2022_Pancreas/Group3/Cells3.csv",
        "samples": "Data_Hwang2022_Pancreas/Samples.csv",
        "units": "UMI"
    },
    "Lin2020": {
        "matrix": "Data_Lin2020_Pancreas/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Lin2020_Pancreas/Genes.txt",
        "cells":  "Data_Lin2020_Pancreas/Cells.csv",
        "samples": "Data_Lin2020_Pancreas/Samples.csv",
        "units": "UMI"
    },
    "Moncada2020": {
        "matrix": "Data_Moncada2020_Pancreas/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Moncada2020_Pancreas/Genes.txt",
        "cells":  "Data_Moncada2020_Pancreas/Cells.csv",
        "samples": "Data_Moncada2020_Pancreas/Samples.csv",
        "units": "UMI"
    },
    "Peng2019": {
        "matrix": "Data_Peng2019_Pancreas/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Peng2019_Pancreas/Genes.txt",
        "cells":  "Data_Peng2019_Pancreas/Cells.csv",
        "samples": "Data_Peng2019_Pancreas/Samples.csv",
        "units": "UMI"
    },
    "Raghavan2021": {
        "matrix": "Data_Raghavan2021_Pancreas/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Raghavan2021_Pancreas/Genes.txt",
        "cells":  "Data_Raghavan2021_Pancreas/Cells.csv",
        "samples": "Data_Raghavan2021_Pancreas/Samples.csv",
        "units": "UMI"
    },
    "Steele2020": {
        "matrix": "Data_Steele2020_Pancreas/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Steele2020_Pancreas/Genes.txt",
        "cells":  "Data_Steele2020_Pancreas/Cells.csv",
        "samples": "Data_Steele2020_Pancreas/Samples.csv",
        "units": "UMI"
    }
}

In [4]:
study_name = 'Hwang2022_Group1'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# type col should exist
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Pancreas'   # PER STUDY CHANGE
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalization
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# convert object cols and save
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 100,000 cells × 22,164 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Pancreatic Ductal Adenocarcinoma']
After gene filter: 97,218 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.40
Detected doublet rate = 0.4%
Estimated detectable doublet fraction = 55.5%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.8%
Elapsed time: 236.9 seconds
After doublet removal: 96,782 cells
After MT filter: 96,773 cells
Normalised (UMI) – max value 11.90


In [5]:
study_name = 'Hwang2022_Group2'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# type col should exist
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Pancreas'   # PER STUDY CHANGE
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalization
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# convert object cols and save
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 100,000 cells × 22,164 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Pancreatic Ductal Adenocarcinoma']
After gene filter: 97,992 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.57
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 46.6%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.3%
Elapsed time: 245.3 seconds
After doublet removal: 97,854 cells
After MT filter: 97,700 cells
Normalised (UMI) – max value 11.95


In [6]:
study_name = 'Hwang2022_Group3'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# type col should exist
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Pancreas'   # PER STUDY CHANGE
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalization
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# convert object cols and save
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 24,988 cells × 22,164 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Pancreatic Ductal Adenocarcinoma']
After gene filter: 24,329 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.31
Detected doublet rate = 0.7%
Estimated detectable doublet fraction = 57.2%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 1.2%
Elapsed time: 26.9 seconds
After doublet removal: 24,160 cells
After MT filter: 24,160 cells
Normalised (UMI) – max value 11.63


In [7]:
study_name = 'Lin2020'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# type col should exist
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Pancreas'   # PER STUDY CHANGE
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalization
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# convert object cols and save
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 14,926 cells × 22,217 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Pancreatic Ductal Adenocarcinoma']
After gene filter: 14,854 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.50
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 41.9%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.4%
Elapsed time: 9.7 seconds
After doublet removal: 14,832 cells
After MT filter: 14,832 cells
Normalised (UMI) – max value 13.31


In [8]:
study_name = 'Moncada2020'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# type col should exist
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Pancreas'   # PER STUDY CHANGE
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalization
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# convert object cols and save
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 3,659 cells × 19,738 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Pancreatic Ductal Adenocarcinoma']
After gene filter: 3,638 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.49
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 10.0%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 1.4%
Elapsed time: 2.2 seconds
After doublet removal: 3,633 cells
After MT filter: 3,633 cells
Normalised (UMI) – max value 12.92


In [9]:
study_name = 'Peng2019'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# type col should exist
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Pancreas'   # PER STUDY CHANGE
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalization
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# convert object cols and save
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 27,953 cells × 18,302 genes
   cell_type: 1446 NaN
   cell_type: 1446 suspicious
Flagged 1446 cells
After metadata drop: 26,507 cells
   Added cancer_type column. Unique values: ['Pancreatic Ductal Adenocarcinoma']
After gene filter: 25,993 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.64
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 11.9%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.4%
Elapsed time: 31.1 seconds
After doublet removal: 25,980 cells
After MT filter: 25,980 cells
Normalised (UMI) – max value 13.44


In [10]:
study_name = 'Raghavan2021'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# type col should exist
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Pancreas'   # PER STUDY CHANGE
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalization
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# convert object cols and save
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 24,412 cells × 16,814 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Pancreatic Ductal Adenocarcinoma' 'Neuroendocrine Tumor']
After gene filter: 23,857 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.72
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 0.9%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.0%
Elapsed time: 23.6 seconds
After doublet removal: 23,857 cells
After MT filter: 18,077 cells
Normalised (UMI) – max value 13.15


In [11]:
study_name = 'Steele2020'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# type col should exist
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Pancreas'   # PER STUDY CHANGE
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalization
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# convert object cols and save
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 48,570 cells × 32,738 genes
   cell_type: 10585 NaN
   cell_type: 10585 suspicious
Flagged 10585 cells
After metadata drop: 37,985 cells
   Added cancer_type column. Unique values: ['Pancreatic Ductal Adenocarcinoma']
After gene filter: 34,549 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.74
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 4.1%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.3%
Elapsed time: 76.0 seconds
After doublet removal: 34,545 cells
After MT filter: 31,460 cells
Normalised (UMI) – max value 13.61


In [12]:
INPUT_PATTERN = "*.h5ad"   

files = sorted(glob.glob(INPUT_PATTERN))

for f in files:
    adata = sc.read_h5ad(f)
    study_name = adata.obs['study'].iloc[0] if 'study' in adata.obs.columns else f
    print(f"{study_name}  –  {adata.n_obs:,} cells, {adata.n_vars:,} genes")
    
    for col in sorted(adata.obs.columns):
        if pd.api.types.is_numeric_dtype(adata.obs[col]):
            continue
        
        vals = adata.obs[col].dropna().unique()
        n_unique = len(vals)
        
        if n_unique <= 25:
            val_str = ', '.join(str(v) for v in sorted(vals))
        else:
            val_str = f"{n_unique} unique values – first 15: " + ', '.join(str(v) for v in sorted(vals)[:15])
        
        print(f" {col}: {val_str}")
    
    print()

Hwang2022_Group1  –  96,773 cells, 22,164 genes
 cancer_type: Pancreatic Ductal Adenocarcinoma
 cell_cycle_phase: G1/S, G2/M, Intermediate, Not cycling, nan
 cell_subtype: 30 unique values – first 15: ADM, Acinar, Adipocyte, Alpha, B_cell, Beta, CAF, CD4+ T, CD8+ T, Delta, Dendritic, Ductal, Ductal (atypical), Epsilon, Gamma
 cell_type: Adipocyte, B_cell, Dendritic, Endocrine, Endothelial, Epithelial, Fibroblast, Macrophage, Malignant, Mast, Myocyte, NK_cell, Neuron, Neutrophil, Pericyte, Plasma, Schwann, T_cell
 disease: PDAC
 mp_assignment: 64 unique values – first 15: B-cells1, CAF1, CAF2, CAF3, CAF4, CAF8, CAF9, CD4 - Cytotoxic, CD4 - Interferon, CD4 - Treg, CD8 - Chromatin, CD8 - Naive2, CD8 - Naive3, CD8 - Unassigned2, Cell Cycle
 mp_top: 127 unique values – first 15: Alveolar, Androgen-prostate, Astrocytes, B-cells1, CAF1, CAF10, CAF2, CAF3, CAF4, CAF5, CAF6, CAF7, CAF8, CAF9, CD4 - Cell Cycle
 sample: T16, T18, T22, T23, T25, T4, T5, T7, U10, U11, U12, U15, U16, U18, U2, U3, U5

In [ ]:
INPUT_PATTERN = "*.h5ad"
OUTPUT_FILE = "pancreas.h5ad"

valid_prefixes = [
    'Hwang2022_Group1', 'Hwang2022_Group2', 'Hwang2022_Group3',
    'Lin2020', 'Moncada2020', 'Peng2019', 'Raghavan2021', 'Steele2020'
]
all_files = sorted(glob.glob(INPUT_PATTERN))
real_files = [f for f in all_files if any(f.startswith(p) for p in valid_prefixes)]
print(f"Found {len(real_files)} study files (ignored {len(all_files)-len(real_files)})\n")

def label_malignant(adata, study_name):
    if 'cell_type' in adata.obs.columns:
        ct = adata.obs['cell_type'].astype(str).str.lower().str.strip()
        if ct.str.contains('malignant', na=False).any():
            return ct.str.contains('malignant', na=False), "cell_type='Malignant'"
    return pd.Series(False, index=adata.obs.index), "all non-malignant"

temp_files = []
total_malig = 0
total_non = 0

for f in real_files:
    adata = sc.read_h5ad(f)
    study_name = adata.obs['study'].iloc[0]

    if 'cancer_type' not in adata.obs.columns or adata.obs['cancer_type'].isna().all():
        adata.obs['cancer_type'] = 'Pancreatic Cancer'
    else:
        adata.obs['cancer_type'] = adata.obs['cancer_type'].astype(str).fillna('Pancreatic Cancer')

    malignant_mask, strategy = label_malignant(adata, study_name)
    adata.obs['is_malignant'] = malignant_mask.astype(int)

    malig_count = malignant_mask.sum()
    non_count = (~malignant_mask).sum()
    total_malig += malig_count
    total_non += non_count

    print(f"{study_name:<25s}  {adata.n_obs:>7,} cells  →  malignant: {malig_count:>6,}   non‑malignant: {non_count:>7,}   ({strategy})")

    temp_file = f"__temp_{study_name}.h5ad"
    adata.write_h5ad(temp_file)
    temp_files.append(temp_file)

print(f"\n{'='*60}")
print(f"TOTAL: {total_malig+total_non:>7,} cells  →  malignant: {total_malig:>6,}   non‑malignant: {total_non:>7,}")

print(f"\nMerging {len(temp_files)} studies...")
adatas = [sc.read_h5ad(tf) for tf in temp_files]
adata_all = adatas[0].concatenate(adatas[1:], batch_key='study', join='inner', index_unique='-')
print(f"Merged: {adata_all.n_obs:,} cells × {adata_all.n_vars:,} genes")

study_names_original = []
for tf in temp_files:
    ad = sc.read_h5ad(tf)
    study_names_original.append(ad.obs['study'].iloc[0])
study_map = {str(i): name for i, name in enumerate(study_names_original)}
adata_all.obs['study'] = adata_all.obs['study'].astype(str).map(study_map)

essential = ['sample', 'cell_type', 'cancer_type', 'study', 'is_malignant']
keep = [c for c in essential if c in adata_all.obs.columns]
adata_all.obs = adata_all.obs[keep]

print("\nCancer types:", sorted(adata_all.obs['cancer_type'].astype(str).unique()))
print("Malignant distribution:")
print(adata_all.obs['is_malignant'].value_counts())

for col in keep:
    n_nan = adata_all.obs[col].isna().sum()
    if n_nan > 0:
        adata_all = adata_all[~adata_all.obs[col].isna(), :].copy()

for col in adata_all.obs.columns:
    if adata_all.obs[col].dtype == 'object':
        adata_all.obs[col] = adata_all.obs[col].astype(str)

print(f"Final clean dataset: {adata_all.n_obs:,} cells × {adata_all.n_vars:,} genes")
adata_all.write_h5ad(OUTPUT_FILE)

for tf in temp_files:
    os.remove(tf)

Found 8 study files (ignored 0)

Hwang2022_Group1            96,773 cells  →  malignant: 31,910   non‑malignant:  64,863   (cell_type='Malignant')
Hwang2022_Group2            97,700 cells  →  malignant: 30,330   non‑malignant:  67,370   (cell_type='Malignant')
Hwang2022_Group3            24,160 cells  →  malignant:  1,428   non‑malignant:  22,732   (cell_type='Malignant')
Lin2020                     14,832 cells  →  malignant: 10,909   non‑malignant:   3,923   (cell_type='Malignant')
Moncada2020                  3,633 cells  →  malignant:    634   non‑malignant:   2,999   (cell_type='Malignant')
Peng2019                    25,980 cells  →  malignant:  7,607   non‑malignant:  18,373   (cell_type='Malignant')
Raghavan2021                18,077 cells  →  malignant:  5,098   non‑malignant:  12,979   (cell_type='Malignant')
Steele2020                  31,460 cells  →  malignant:  4,816   non‑malignant:  26,644   (cell_type='Malignant')

TOTAL: 312,615 cells  →  malignant: 92,732   non‑malig

/tmp/ipykernel_42252/563587387.py:51: FutureWarning: The method concatenate is deprecated and will be removed in the future. Use anndata.concat instead of AnnData.concatenate. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_all = adatas[0].concatenate(adatas[1:], batch_key='study', join='inner', index_unique='-')


Merged: 312,615 cells × 14,531 genes

Cancer types: ['Neuroendocrine Tumor', 'Pancreatic Ductal Adenocarcinoma']
Malignant distribution:
is_malignant
0    219883
1     92732
Name: count, dtype: int64
Final clean dataset: 312,615 cells × 14,531 genes
